Kacper Kaszuba Lab4 16618

# Laboratorium 1: Dekoratory, Deskryptory i Generatory
### Skoroszyt główny

---

## Cele Laboratorium
Celem dzisiejszych zajęć jest opanowanie zaawansowanych konstrukcji języka Python, które są niezbędne do projektowania nowoczesnej architektury aplikacji.

### System Wspomagania AI (Tutor)
W trakcie rozwiązywania zadań możesz korzystać z pomocy dedykowanego tutora AI. System oferuje 6 poziomów wsparcia:
1. **Ogólna wskazówka**: Sugestia kierunku rozwiązania.
2. **Pseudokod**: Logiczny opis algorytmu.
3. **Mały fragment kodu**: Kluczowa linia lub konstrukcja.
4. **Częściowa implementacja**: Szkielet kodu do uzupełnienia.
5. **Szczegółowe wyjaśnienie**: Analiza mechanizmu działania.
6. **Pełne rozwiązanie**: Dostępne w sytuacjach ostatecznych.

---

## 1. Dekoratory

### DEMO: Dekorator @timer 
Stwórz dekorator @timer, który będzie mierzył i wyświetlał czas wykonania funkcji.

In [1]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Czas wykonania {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper

@timer
def example_task():
    time.sleep(0.5)
    print("Zadanie zakończone.")

example_task()

Zadanie zakończone.
Czas wykonania example_task: 0.5010 s


### Zadanie 1: Liczba elementów listy
Stwórz dekorator, który będzie odpowiedzialny za wyświetlanie liczby elementów listy, jeśli jakakolwiek lista pojawi się w parametrach funkcji dekorowanej. 

**Protip:** użyj isinstance do sprawdzenia czy parametr jest listą. Pamiętaj o zachowaniu metadanych funkcji.

In [6]:
from unicodedata import name
import functools

# TODO: Implementacja dekoratora
def show_list_length(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        for arg in args:
            if isinstance(arg, list):
                print(f"Znaleziono instancje listy w args, długość listy to: {len(arg)}")

        for value in kwargs.values():
            if isinstance(value, list):
                print(f"Znaleizono isntacje lsity w kwargs, długość listy to: {len(value)}")

        result = func(*args, **kwargs)
        return result
    return wrapper

# Test:
@show_list_length
def process_data(data_list, name):
    print(f"Przetwarzanie {name}")

process_data([1, 2, 3, 4], "Lista")
process_data(data_list=["a", "b"], name="Słownik")

Znaleziono instancje listy w args, długość listy to: 4
Przetwarzanie Lista
Znaleizono isntacje lsity w kwargs, długość listy to: 2
Przetwarzanie Słownik


### Zadanie 2: Logowanie do pliku
Stwórz dekorator, który będzie zapisywał w pliku *.log nazwę funkcji dekorowanej, datę oraz długość wykonania. Nazwa pliku będzie podana jako argument dekoratora.

**Protip:** użyj biblioteki datetime. Pamiętaj o tym, żeby dekoratory przyjęły metadanych funkcji dekorującej.

In [11]:
import functools
from datetime import datetime
import time



# TODO: Implementacja dekoratora z argumentem
def logger(filename):
    def save_log(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # data
            date = datetime.now()
            # czas
            start_time = time.perf_counter()
            result = func(*args, **kwargs)
            end_time = time.perf_counter()
            func_time = end_time - start_time
            with open(f"{filename}.log", 'a', encoding='utf-8') as log:
                log.write(f"File name: {func.__name__}\nDate: {date}\nTime of func: {func_time}")
            
            return  result
        return wrapper
    return save_log


@logger("testowe_logi")
def test_func():
    time.sleep(1)
    print("Test log")

test_func()

Test log


--- 
## 2. Deskryptory

### DEMO: Walidator e-mail klasy Student
Stwórz deskryptor, który będzie działał jako walidator email klasy Student. Klasa Student zawiera pola imie, nazwisko i email. Deskryptor ten powinien sprawdzać poprawność danych wprowadzanych podczas tworzenia lub modyfikowania instancji Student.

In [ ]:
class EmailValidator:
    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if "@" not in value:
            raise ValueError(f"Błędny format adresu email: {value}")
        instance.__dict__[self.name] = value

class Student:
    email = EmailValidator()
    
    def __init__(self, imie, nazwisko, email):
        self.imie = imie
        self.nazwisko = nazwisko
        self.email = email

try:
    s = Student("Jan", "Kowalski", "jan.kowalski@wsei.edu.pl")
    print(f"Utworzono studenta: {s.email}")
    # s.email = "invalid_at_email" # Powinno rzucić błąd
except ValueError as e:
    print(e)

### Zadanie 3: Rejestrowanie dostępu
Stwórz klasę Uzytkownik. Klasa powinna zawierać atrybuty imie i wiek. Opracuj deskryptor, który będzie rejestrował dostęp do tych atrybutów za pomocą logowania. Deskryptor powinien logować informacje o odczycie (__get__) oraz zapisie (__set__) wartości atrybutu.

In [13]:
# TODO: Implementacja deskryptora logującego dostęp
class AccessLogger:
    def __set_name__(self, owner, name):
        self.nazwa_prywatna = "_" + name

    def __get__(self, instance, owner):
        print(f"Odczytanio atrybut {self.nazwa_prywatna}")
        return instance.__dict__[self.nazwa_prywatna]

    def __set__(self, instance, value):
        print(f"Zmieniono atrybut {self.nazwa_prywatna} na wartość {value}")
        instance.__dict__[self.nazwa_prywatna] = value

class Uzytkownik:
    imie = AccessLogger()
    wiek = AccessLogger()

    def __init__(self, startowe_imie, startowy_wiek):
        self.imie = startowe_imie
        self.wiek = startowy_wiek


user = Uzytkownik('Marek', 25)
print(user.imie, "\n")
user.wiek = 26

print(user.wiek)

Zmieniono atrybut _imie na wartość Marek
Zmieniono atrybut _wiek na wartość 25
Odczytanio atrybut _imie
Marek 

Zmieniono atrybut _wiek na wartość 26
Odczytanio atrybut _wiek
26


--- 
## 3. Generatory i Iteratory

### DEMO: Generator Fibonacciego
Napisz klasę, która będzie implementowała generator ciągu Fibonacciego za pomocą metod magicznych __iter__() i __next__().

In [ ]:
class FibonacciGenerator:
    def __init__(self, limit):
        self.limit = limit
        self.a, self.b = 0, 1
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

fib = FibonacciGenerator(10)
print(list(fib))

### Zadanie 4: Generator ciągu Collatza
Opracuj generator ciągu Collatza. Dla liczby naturalnej n, jeśli n jest parzyste, dziel przez 2; jeśli n jest nieparzyste, pomnóż przez 3 i dodaj 1, zaczynając od określonej liczby początkowej, aż do osiągnięcia wartości 1.

In [14]:
# TODO: Implementacja generatora ciągu Collatza
def collatz_generator(n):
    yield n 
    while n > 1:
        if n % 2 == 0:
            n = n // 2
        else:
            n = 3 * n + 1 
        
        yield n


# Test:
for status in collatz_generator(10):
    print(status)

10
5
16
8
4
2
1


---

## Zadania do zrobienia w domu

Poniższe zadania stanowią rozszerzenie materiału i są przeznaczone dla osób chcących zgłębić temat zaawansowanych konstrukcji języka Python.

### Zadanie dodatkowe 1: Dekorator z autoryzacją
Stwórz dekorator `@require_role(role)`, który przyjmuje nazwę wymaganej roli jako argument. Dekorator powinien sprawdzać, czy w globalnym słowniku `current_user` klucz `role` jest zgodny z wymaganym. Jeśli nie, rzuć `PermissionError`.

In [21]:
current_user = {"username": "admin", "role": "superuser"}

# TODO: Implementacja dekoratora @require_role
def require_role(role):
    def autorize(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if current_user["role"] != role:
                raise PermissionError("Zła rola brak  autoryzacji!")
            else:
                return func(*args, **kwargs)

        return wrapper
    return autorize

@require_role("superuser")
def usun_z_bazy():
    print("Usunięto z bazy danych")

@require_role("intern")
def wypisz_obowiazki():
    print("Wypisuje obowiazki")


usun_z_bazy() # Działa  
wypisz_obowiazki() # Nie dizała   


Usunięto z bazy danych


PermissionError: Zła rola brak  autoryzacji!

### Zadanie dodatkowe 2: Deskryptor z walidacją typu
Stwórz deskryptor `Typed`, który przyjmuje typ danych (np. `int`, `str`) w konstruktorze. Deskryptor powinien upewnić się, że zapisywana wartość jest tego typu. Jeśli nie, rzuć `TypeError`.

In [34]:
# TODO: Implementacja deskryptora Typed
class Typed:
    def  __init__(self, oczekiwany_typ):
        self.oczekiwany_typ = oczekiwany_typ

    def __set_name__(self, owner, name):
        self.nazwa_prywatna = "_" + name
    
    def __get__(self, instance, owner):
        print(f"Odczytanio atrybut {self.nazwa_prywatna}")
        return instance.__dict__[self.nazwa_prywatna]
        


    def __set__(self, instance, value):
        if not isinstance(value, self.oczekiwany_typ):
            raise  TypeError("Wrtość ma zły typ")
        
        print(f"Zmieniono atrybut {self.nazwa_prywatna} na wartość {value}")
        instance.__dict__[self.nazwa_prywatna] = value


class  Produkt:
    nazwa = Typed(str)
    cena = Typed(float)

    def __init__(self, nazwa, cena):
        self.nazwa = nazwa
        self.cena = cena
    
p1 =  Produkt("IPhone", 5030.50)
print(p1.cena)
print(p1.nazwa)

#To bedzie źle
# p2 = Produkt("Laptop", 6000)
# p3 = Produkt(123, 500.3)




Zmieniono atrybut _nazwa na wartość IPhone
Zmieniono atrybut _cena na wartość 5030.5
Odczytanio atrybut _cena
5030.5
Odczytanio atrybut _nazwa
IPhone


### Zadanie dodatkowe 3: Nieskończony generator liczb pierwszych
Opracuj generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie użyj wyrażenia generatorowego, aby stworzyć iterator zwracający tylko te liczby pierwsze, które kończą się cyfrą 7.

In [33]:
def if_prime(n):
    if n < 2:
        return False
    for dzielnik in range(2, n-1):
        if n % dzielnik ==0:
            return False
    return True

# TODO: Implementacja generatora liczb pierwszych
def prime_generator():
    n = 2
    while True:
        if if_prime(n):
            yield n
        n += 1 


pierwsze_z_siedem = (liczba for liczba in prime_generator() if liczba % 10 == 7)

for _ in range(5):
    print(next(pierwsze_z_siedem))

7
17
37
47
67
